<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 24 · Machine Learning Workflow for Fixed Income

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch24_machine_learning_workflow_fixed_income.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Machine learning helps in fixed income only when the workflow respects time. You will trace the companion script's leakage controls end to end: a chronological 70/30 split of a 12-row weekly panel, purging of the training row whose label ends on the first test date, feature scaling fitted on training rows only, and a small ridge regression with an unpenalized intercept. You will compare the ridge forecast with a no-skill baseline on four held-out weeks, read a model card built from illustrative metrics, and practice stating the target horizon and feature timestamps before quoting any accuracy number.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script sorts the 12-row weekly feature panel by date, splits chronologically at 70 percent, purges the training row whose label reaches the first test date, and fits a small ridge regression with training-only scaling. The printed comparison gives ridge a mean absolute error of 0.0989 against the baseline's 0.1114 on the four held-out weeks, with identical directional hit rates of 0.75.

In [ ]:
SCRIPT = ROOT / "code" / "ch24_machine_learning_workflow_fixed_income.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Check the workflow controls, not just the metrics.

- The split is chronological: the first 8 rows train and the last 4 weekly dates test; after purging, 7 training rows remain.
- The purge removes the training row whose target end date falls on the first test date, so no training label overlaps the test window.
- Scaling uses training means and standard deviations only, and the intercept is not penalized.
- Both models post a hit rate of 0.75 on 4 test rows; the ridge edge is the MAE of 0.0989 versus 0.1114, and four observations cannot make that persuasive.
- The panel is synthetic teaching data, and the stored model-comparison table is a separate illustrative snapshot, not this fit's holdout results.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch24_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Make the time ordering explicit and build a model-card example from the separate illustrative metric snapshot. Its numbers are not the fitted companion script's holdout results.

In [ ]:
panel = pd.read_csv(DATA / "ch24_ml_feature_panel.csv", parse_dates=["date"])
comparison = pd.read_csv(DATA / "ch24_model_comparison.csv")
chronological = panel.date.is_monotonic_increasing
model_card = pd.DataFrame({
    "business_question": ["one-step curve target"],
    "target_horizon": ["next observation"],
    "split_rule": ["chronological"],
    "baseline": [comparison.model.iloc[0]],
    "illustrative_lowest_mae": [
        comparison.sort_values("mae").model.iloc[0]
    ],
    "dates_in_order": [chronological],
})
model_card

## Mini-lab: A chronological split with the warm-up row purged

A seeded 12-row weekly panel is split at 70 percent by row count, holding out the last 4 weeks. The target is the change in the curve level to the next weekly observation, which is known only at the label end date. One training row's label ends exactly on the first test date; keeping it would let training touch test-period information, so the row is purged, exactly as the chapter script's split does. The level path is seeded and the features are observable at the row date.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(5)  # seed
dates = pd.date_range("2026-01-02", periods=12, freq="W-FRI")
level = 3.0 + np.cumsum(rng.normal(0.0, 0.05, 12))
panel = pd.DataFrame({"date": dates, "level": level})
panel["target"] = panel.level.shift(-1) - panel.level
panel["label_end"] = panel.date.shift(-1)  # when target is known
split = int(0.7 * len(panel))
test = panel.iloc[split:]
train = panel.iloc[:split]
train = train.loc[train["label_end"] < test["date"].iloc[0]]  # purge
print(f"train rows: {len(train)}  test rows: {len(test)}")
print(f"last label end: {train['label_end'].iloc[-1].date()}")
print(f"first test date: {test['date'].iloc[0].date()}")

### What to observe

The 70 percent boundary leaves 8 candidate training rows and 4 test rows, but the row whose label ends on 2026-02-27, the first test date, is purged, leaving 7 training rows. The last surviving training label ends on 2026-02-20, strictly before the test window opens.

## Exercises

Complete these with the printed script output; name the control each task exercises.

1. **Audit the split.** Confirm the split is chronological and that 7, not 8, training rows survive the purge. *Hint:* Compare each training row's target end date with the first test date.
2. **Judge the metric.** Recompute MAE and hit rate for both models and state whether the ridge edge on 4 rows is persuasive. *Hint:* Four holdout observations cannot support a strong claim.
3. **Find the leakage.** Explain what would go wrong if the scaler were fitted on the full panel or the rows were shuffled before splitting. *Hint:* Test-period statistics would then shape the training inputs.
4. **Write the model card.** Draft a card naming the business question, target horizon, split rule, baseline, and metrics from the printed output. *Hint:* Use the chapter's card fields rather than invented performance claims.
5. **Separate the snapshots.** Contrast the script's printed MAE of 0.0989 with the stored comparison table's 0.0587 and say which one describes the fitted model. *Hint:* The stored table is a hand-constructed teaching snapshot.

For more detail, see Chapter 24 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">